# Snow and water in the mountains

## 1 · Prepare your workspace and access

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">📁 STEP 1 — UPLOAD THE NOTEBOOK AND STUDY AREA</div>
  <p style="margin: 0 0 8px; color: #493600;">In JupyterLab, open a folder under <strong>mystorage</strong> and upload this notebook there. For Oetztal, also upload the supplied <code>oetztal_euhydro.gpkg</code> into the <strong>same folder</strong>; ask the facilitator if it is missing.</p><p style="margin: 0 0 8px; color: #493600;">For a custom area, upload your own AOI file or supply an inline GeoJSON geometry in section 3.</p>
</div>

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">⚙️ STEP 2 — SELECT THE GEO SCIENCE KERNEL</div>
  <p style="margin: 0 0 8px; color: #493600;">Open the notebook. Click the kernel name at the <strong>top right</strong> and select the <strong>Geo Science</strong> kernel provided by CDSE JupyterLab.</p>
</div>

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">🔐 STEP 3 — CREATE YOUR PRIVATE CREDENTIALS FILE</div>
  <p style="margin: 0 0 8px; color: #493600;">Choose <strong>File → New → Text File</strong>. Save it as <code>credentials.ini</code> in the <strong>same folder as this notebook</strong>. If it already exists, update it rather than creating another file.</p><p style="margin: 0 0 8px; color: #493600;">Copy the template below into the text file, replace the placeholders with your own CDSE username and password, and save it. <strong>Never put real credentials in a notebook cell or share your completed credentials file.</strong></p><pre style="background-color: #fff8e1; color: #493600; padding: 12px; white-space: pre-wrap;"><code>[CDSE]
username = YOUR_USERNAME
password = YOUR_PASSWORD</code></pre>
</div>

The notebook uses CDSE OData and this login for DEM and snow downloads. Monthly water is available only from the public HRWSI bucket and does not use this login. Downloaded snow rasters are cached in the visible `cdse-assets/` folder beside the notebook to avoid repeated transfers.

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">⏸️ SETUP COMPLETE — PAUSE HERE</div>
  <p style="margin: 0 0 8px; color: #493600;"><strong>Complete setup, then pause here while we introduce the exercise and datasets.</strong> You do not need to run the notebook yet.</p>
  <p style="margin: 0; color: #493600;">When the hands-on exercise starts, choose <strong>Run → Run All Cells</strong>. Execution deliberately stops at checkpoint 1 in section 3. Only the configuration and daily-date checkpoints require code edits.</p>
</div>

## 2 · About the exercise

### 2.1 · What are we investigating?

In this exercise, we explore how snow and mapped surface water change through the season in a mountain catchment. The live demonstration uses **Oetztal in 2019**.

Our questions are:

- **Snow through the season:** how do fractional snow cover and radar wet-snow detections differ between spring and summer?
- **Snow and elevation:** when and at which elevations is wet snow detected?
- **Snow and water together:** how does the timing of snow change compare with monthly mapped surface water?

We first locate the study area and inspect terrain and product availability. We then compare two daily snow maps, examine wet snow by elevation, and compare seasonal snow patterns with monthly water. Valid observation coverage matters throughout: missing pixels are not zeroes.

### 2.2 · Main products: snow and water

These are the **HR-WSI products** we explore in this exercise.

| Product | What it shows | Native pixel spacing | Temporal information | Coordinate reference system |
|---|---|---|---|---|
| Fractional snow cover (FSC) | Snow fraction from Sentinel-2 optical observations | 20 m | Individual acquisition dates; availability varies, and cloud and other masks can limit usable pixels | WGS 84 / UTM, by tile zone |
| SAR wet snow (SWS) | Wet or dry/no/patchy snow from Sentinel-1 radar observations | 60 m | Individual acquisition dates; availability and spatial coverage depend on acquisitions and masks | WGS 84 / UTM, by tile zone |
| Monthly water (WLRA) | Detected surface water | 10 m | Monthly products, rather than individual daily observations | ETRS89 / LAEA Europe (EPSG:3035) |

**Fractional Snow Cover (FSC)**

FSC uses Sentinel-2 optical observations to estimate the percentage of each 20 m pixel covered by snow. This notebook uses the **top-of-canopy layer (FSCTOC)**: it describes snow visible above the canopy, not a direct measurement of snow beneath trees. Values range from 0% (no detected snow) to 100% (fully snow-covered). Clouds, shadows and missing acquisitions can limit usable coverage; cloud or no-data pixels are not zero snow. The accompanying FSCTOC quality layer describes confidence from 0 (high) to 3 (minimal).

**SAR Wet Snow (SWS)**

SWS uses Sentinel-1 radar observations to map wet snow in selected high-mountain areas at 60 m pixel spacing. Wet snow contains liquid water within the snowpack. The notebook uses the **Wet Snow in high Mountain areas (WSM)** classification: wet snow is distinguished from a combined dry snow / no snow / patchy snow class; those three conditions cannot be separated by this product. Forest, water, urban and non-mountain areas, and areas affected by radar geometry, may be masked. The WSM quality layer uses classes 0 (high) to 3 (minimal) and applies to detected wet-snow pixels.

**Monthly water masks (WLRA)**

The **Water Layer Rolling Archive (WLRA)** contains seasonal and monthly binary water masks. This notebook uses its **monthly masks**, derived from Sentinel-1 and Sentinel-2 observations at 10 m pixel spacing. A valid pixel indicates whether surface water was detected during the month; it is not a daily observation, water depth or water volume. Unavailable pixels are not dry pixels. For comparisons across months, the notebook uses common valid support and withholds comparable area estimates when required inputs are incomplete.

**Access:** Monthly water masks are available only through the [public HRWSI bucket](https://s3.waw3-2.cloudferro.com/HRWSI) and require no login. This notebook discovers and reads them directly from the bucket. For general HR-WSI bucket-access instructions and download examples, see the [public Python client repository](https://github.com/eea/clms-hrwsi-api-client-python); its README does not specifically document WLRA monthly masks.

### 2.3 · Auxiliary data

These datasets support the analysis; they are not HR-WSI products.

- **Copernicus DEM GLO-30** provides terrain elevation for the hillshade and elevation bands. It is a digital surface model with approximately 30 m spacing, not a time series for the selected analysis period.
- **EU-Hydro** provides the drainage units used to derive the Oetztal study area. The resulting boundary approximates the Tumpen Gauge catchment; it is not an independently verified catchment delineation.

<details><summary>Sources and attribution</summary>

**HR-WSI snow and water**

- [Snow product user manual](https://land.copernicus.eu/en/technical-library/product-user-manual-high-resolution-snow-products-europe/@@download/file)
- [Water product user manual](https://land.copernicus.eu/en/technical-library/product-user-manual-high-resolution-water-products-europe-1/@@download/file)

**Auxiliary data**

- **[EU-Hydro](https://www.eea.europa.eu/en/datahub/datahubitem-view/2e782ca5-c7b2-4b48-8928-03031b642176):** European Environment Agency. Generated using European Union’s Copernicus Land Monitoring Service information; drainage polygons were selected and combined for this study area.
- **Copernicus DEM GLO-30:** release 2024_1 · [DOI](https://doi.org/10.5270/ESA-c5d3d65) · [Licence](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/DEM/resources/license/License-COPDEM-30.pdf)

  Produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; all rights reserved. The organisations in charge of the Copernicus programme by law or by delegation do not incur any liability for any use of the Copernicus WorldDEM-30.

</details>

In [ ]:
# RUN ONLY — no changes needed
import os

# CDSE may inherit paths to an older conda PROJ database.
# Let pip-installed geospatial packages use their bundled PROJ data.
# Run before importing geopandas, rasterio, or pyproj.
os.environ.pop("PROJ_LIB", None)
os.environ.pop("PROJ_DATA", None)

import math
from collections.abc import Callable, Iterator
from typing import Any

import folium
import geopandas as gpd
from shapely.geometry import box

## 3 · Choose your analysis — checkpoint 1

The configuration has two options: `oetztal` uses the supplied GeoPackage, an approximation of the Tumpen Gauge catchment; `custom` accepts a polygon file with a defined CRS or an inline GeoJSON Polygon, MultiPolygon or Feature in EPSG:4326 (longitude, latitude). Exactly one AOI source is required.

The date window includes the start and excludes the end. Samples per month are maximum numbers of observation dates per product and month; fewer may be available. Use `None` to retain all available dates. The two daily-map dates at checkpoint 2 are chosen independently of this sampling. Catalogue availability does not guarantee valid pixels inside the study area.

<details><summary>How was the Oetztal study area derived?</summary>

The Oetztal study area was derived from [EU-Hydro](https://www.eea.europa.eu/en/datahub/datahubitem-view/2e782ca5-c7b2-4b48-8928-03031b642176) drainage units. It approximates the Tumpen Gauge catchment but is not an independently verified catchment delineation. Its area is approximately 797 km², calculated in ETRS89 / LAEA Europe (EPSG:3035).

See the sources and attribution in section 2.

</details>

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">✏️ EDIT THE CELL BELOW — CONFIGURE YOUR ANALYSIS</div>
  <p style="margin: 0 0 8px; color: #493600;">Follow the arrows: choose <code>"oetztal"</code> or <code>"custom"</code>, then fill in <code>start</code>, <code>end</code> (both <code>YYYY-MM-DD</code>) and <code>samples_per_month</code> in <strong>only the selected preset</strong>. Leave the other preset unchanged.</p><p style="margin: 0 0 8px; color: #493600;"><strong>Suggested live demonstration — Oetztal:</strong> start <strong>2019-01-01</strong>, end <strong>2020-01-01</strong>, with <strong>3 FSC</strong> and <strong>6 SWS samples per month</strong>. Replace each sampling placeholder with a positive whole number, or <code>None</code> for all available dates.</p><p style="margin: 0 0 8px; color: #493600;">For custom, fill in <strong>either</strong> <code>aoi_file</code> <strong>or</strong> <code>geometry</code>; leave the unused source as <code>None</code>.</p><p style="margin: 0 0 8px; color: #493600;"><strong>Then use Run → Run All Cells.</strong> Empty or invalid inputs deliberately stop execution.</p>
</div>

In [ ]:
# EDIT THIS CELL — checkpoint 1: choose and configure your analysis
example = "oetztal"  # <----- CHANGE THIS to "oetztal" or "custom"

presets = {
    "oetztal": {
        "aoi_file": "oetztal_euhydro.gpkg",
        "start": "YYYY-MM-DD",  # <----- ENTER the start date as "YYYY-MM-DD"
        "end": "YYYY-MM-DD",  # <----- ENTER the end date as "YYYY-MM-DD" (exclusive)
        "samples_per_month": {
            "FSC": ...,  # <----- ENTER samples per month
            "SWS": ...,  # <----- ENTER samples per month
        },
    },
    "custom": {
        "aoi_file": None,  # <----- ENTER your AOI file path OR fill in geometry below
        "geometry": None,  # <----- ENTER your GeoJSON geometry or Feature OR use aoi_file above
        "start": "YYYY-MM-DD",  # <----- ENTER the start date as "YYYY-MM-DD"
        "end": "YYYY-MM-DD",  # <----- ENTER the end date as "YYYY-MM-DD" (exclusive)
        "samples_per_month": {
            "FSC": ...,  # <----- ENTER samples per month
            "SWS": ...,  # <----- ENTER samples per month
        },
    },
}

# CHECKPOINT — leave the assertions below unchanged.
from datetime import date
from pathlib import Path

_configuration_help = (
    " See the guidance above, fill in the selected preset in this cell, "
    "then use Run > Run All Cells to continue."
)
assert isinstance(example, str) and example in ("oetztal", "custom"), (
    'Choose example above: "oetztal" or "custom".' + _configuration_help
)
assert isinstance(presets, dict) and isinstance(presets.get(example), dict), (
    "Define a configuration dictionary for the selected example." + _configuration_help
)
preset = presets[example]
assert isinstance(preset.get("samples_per_month"), dict), (
    "Define samples_per_month with FSC and SWS entries." + _configuration_help
)
_missing = [field for field in ("start", "end") if not preset.get(field)]
_missing += [
    f"samples_per_month.{product}" for product in ("FSC", "SWS")
    if preset["samples_per_month"].get(product, ...) is ...
]
assert not _missing, "Fill in: " + ", ".join(_missing) + "." + _configuration_help
for _field in ("start", "end"):
    try:
        _value = preset[_field]
        assert isinstance(_value, str) and date.fromisoformat(_value).isoformat() == _value
    except (AssertionError, TypeError, ValueError):
        raise AssertionError(f"{_field} must be a valid date in YYYY-MM-DD format." + _configuration_help) from None
assert preset["start"] < preset["end"], "Start must be before end." + _configuration_help
for _product in ("FSC", "SWS"):
    _count = preset["samples_per_month"][_product]
    assert _count is None or (type(_count) is int and _count > 0), (
        f"samples_per_month.{_product} must be a positive whole number or None (all available dates)." + _configuration_help
    )
assert bool(preset.get("aoi_file")) != bool(preset.get("geometry")), (
    "Provide exactly one study-area source: aoi_file or geometry." + _configuration_help
)
if preset.get("aoi_file"):
    assert isinstance(preset.get("aoi_file"), str) and Path(preset["aoi_file"]).is_file(), (
        "AOI file not found. Check aoi_file and upload the file beside the notebook, or provide its correct path." + _configuration_help
    )
else:
    from shapely.geometry import shape
    from shapely.errors import GEOSException
    try:
        _geojson = preset["geometry"]
        if _geojson.get("type") == "Feature":
            _geojson = _geojson["geometry"]
        _geometry = shape(_geojson)
        assert _geometry.geom_type in ("Polygon", "MultiPolygon")
        assert not _geometry.is_empty and _geometry.is_valid
        _west, _south, _east, _north = _geometry.bounds
        assert -180 <= _west < _east <= 180 and -90 <= _south < _north <= 90
    except (AssertionError, AttributeError, KeyError, IndexError, TypeError, ValueError, GEOSException):
        raise AssertionError(
            "Fill in custom.geometry with a valid, non-empty GeoJSON Polygon, MultiPolygon or Feature "
            "using longitude, latitude coordinates in EPSG:4326." + _configuration_help
        ) from None
print("Configuration complete. Continue to the daily-date checkpoint in section 5.")

## 4 · Explore the area and available data

Which reference tiles intersect the study area, and when are products available? Strong outlines intersect the AOI; pale outlines are context only. For Oetztal, the marker indicates the approximate Tumpen Gauge location.

<details><summary>Optional technical notes: source grids</summary>

Sentinel reference tiles are from the EEA HR-WSI client MGRS grid. Water cells follow the European 100 km LAEA reference grid; DEM cells follow the nominal one-degree Copernicus DEM grid. Context outlines do not imply availability; discovery uses only the AOI. The DEM is a surface model, not a hydrologically conditioned terrain model.

</details>

In [ ]:
# RUN ONLY — no changes needed
# Load the selected study area and settings.
from pathlib import Path
if example not in presets:
    raise ValueError("Complete checkpoint 1: choose and configure 'oetztal' or 'custom'.")
preset = presets[example]
study_area = example
start_date, end_date = preset["start"], preset["end"]
samples_per_month = preset["samples_per_month"].copy()
if preset.get("aoi_file"):
    aoi_file = preset["aoi_file"]
    if not Path(aoi_file).is_file():
        raise FileNotFoundError("AOI file not found. Check aoi_file and upload the file or provide its correct path.")
    aoi = gpd.read_file(aoi_file).to_crs(4326)
else:
    from shapely.geometry import shape
    try:
        geojson = preset["geometry"]
        if geojson.get("type") == "Feature":
            geojson = geojson["geometry"]
        geometry = shape(geojson)
    except (KeyError, TypeError, ValueError, AttributeError):
        raise ValueError("Provide a valid GeoJSON Polygon or MultiPolygon in the custom preset.") from None
    if geometry.geom_type not in ("Polygon", "MultiPolygon") or geometry.is_empty or not geometry.is_valid:
        raise ValueError("The custom GeoJSON must be a valid, non-empty Polygon or MultiPolygon.")
    west, south, east, north = geometry.bounds
    if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
        raise ValueError("GeoJSON coordinates must be longitude, latitude in EPSG:4326.")
    aoi = gpd.GeoDataFrame(geometry=[geometry], crs=4326)

# Technical settings — no changes needed for the exercise.
example_pass = {"FSC": 0, "SWS": 0}
target_elevation_bands = 16
seed = 42

# European display context, including Iceland, the Azores and the Canary Islands.
# This is a viewing extent, not a product-availability boundary.
map_region = gpd.GeoDataFrame(geometry=[box(-32, 27, 45, 72).segmentize(1)], crs=4326)
url = "https://raw.githubusercontent.com/eea/clms-hrwsi-api-client-python/main/MGRS_tiles.gpkg"
sentinel_grid = gpd.read_file(url, bbox=tuple(map_region.total_bounds)).rename(columns={"Name": "tile"}).to_crs(4326)

# Construct reference cells for discovery and a separate European display context.
def intersecting_cells(
    crs: int, size: float, tile_name: Callable[[int, int], str],
    area: gpd.GeoDataFrame | None = None,
) -> gpd.GeoDataFrame:
    """Return reference-grid cells intersecting an area (the AOI by default)."""
    area = (aoi if area is None else area).to_crs(crs)
    xmin, ymin, xmax, ymax = area.total_bounds
    cells = [(tile_name(x, y), box(x * size, y * size, (x + 1) * size, (y + 1) * size))
             for x in range(math.floor(xmin / size), math.floor(xmax / size) + 1)
             for y in range(math.floor(ymin / size), math.floor(ymax / size) + 1)]
    grid = gpd.GeoDataFrame(cells, columns=["tile", "geometry"], crs=crs)
    return grid[grid.intersects(area.geometry.unary_union)]

def water_tile_name(x: int, y: int) -> str:
    return f"E{x:02d}N{y:02d}"


def dem_tile_name(x: int, y: int) -> str:
    return f"{'N' if y >= 0 else 'S'}{abs(y):02d}_{'E' if x >= 0 else 'W'}{abs(x):03d}"


water = intersecting_cells(3035, 100_000, water_tile_name)

# Context tiles are display-only: never use them to expand data discovery.
water_grid_context = intersecting_cells(3035, 100_000, water_tile_name, area=map_region)
dem_grid_context = intersecting_cells(4326, 1, dem_tile_name, area=map_region)

### 4.1 · Catchment and intersecting tiles

Explore the reference-grid map, then continue through discovery, terrain and availability. No code changes are needed.

In [ ]:
# RUN ONLY — no changes needed
# Esri imagery with complete European reference-grid context.
west, south, east, north = aoi.total_bounds
m = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=9, tiles="Esri.WorldImagery",
)

for data, name, color in [
    (aoi, "Study area", "yellow"),
    (sentinel_grid, "Sentinel-2 tiles (FSC / SWS)", "cyan"),
    (water_grid_context, "Water grid", "#ff78f0"),
    (dem_grid_context, "Copernicus DEM grid", "#ff9e36"),
]:
    if data.empty:
        raise ValueError(f"{name} is empty. Rerun the study-area and grid preparation cell above.")
    display_data = data.to_crs(4326).copy()
    display_data["intersects_aoi"] = display_data.intersects(aoi.to_crs(4326).geometry.unary_union)
    is_grid = "tile" in display_data
    if is_grid:
        display_data = display_data[["tile", "intersects_aoi", "geometry"]]
    layer = folium.FeatureGroup(
        name=name, overlay=True, control=True,
        show=name in ("Study area", "Sentinel-2 tiles (FSC / SWS)"),
    ).add_to(m)
    folium.GeoJson(
        display_data,
        style_function=lambda feature, color=color: {
            "color": color, "fillOpacity": 0,
            "weight": 2 if feature["properties"]["intersects_aoi"] else 1,
            "opacity": 1 if feature["properties"]["intersects_aoi"] else 0.3,
        },
        highlight_function=(lambda feature: {
            "weight": 3, "fillOpacity": 0.08,
            "opacity": 1 if feature["properties"]["intersects_aoi"] else 0.6,
        }) if is_grid else None,
        tooltip=folium.GeoJsonTooltip(
            fields=["tile"], aliases=["Tile ID:"], sticky=True,
            style="font-size: 13px; padding: 6px 8px;",
        ) if is_grid else None,
    ).add_to(layer)

if study_area == "oetztal":
    folium.Marker([47.163333, 10.910833], tooltip="Tumpen Gauge (HZB 201434)",
                  icon=folium.Icon(color="red", icon="tint")).add_to(m)
m.fit_bounds([[south, west], [north, east]])
# Add the control last so every named overlay is included.
folium.LayerControl(collapsed=False).add_to(m)
m

**Why combine tiles?** The study area can cross more than one product tile, and neighbouring tiles may overlap. The notebook combines compatible snow tiles observed close in time into one view of the area.

In an overlap, the notebook sorts tiles **alphabetically by tile ID** and takes the first valid snow value. For example, `T32TPS` comes before `T32TPT`. If the first tile has no valid snow value at a pixel, a valid value from a later tile can fill that pixel. Values are not averaged, and the notebook does not choose the tile with the best quality. The quality value always comes from the same tile as the selected snow value.

This is a consistent selection rule, not a guarantee that the chosen tile is the more accurate one.

<details>
<summary>Optional technical notes: native grids, mosaicking and overlapping tiles</summary>

Snow and QA stay on compatible native grids without resampling; AOI membership uses pixel centres. Incompatible grids stop processing.

**Our mosaicking decisions**

- **Grouping:** handle FSC/SWS and each satellite separately. Sort candidates by sensing timestamp, tile ID, then catalogue product UUID, all ascending. Add distinct tiles only while their timestamps are within **120 seconds of the first tile** (not a rolling window). A repeated tile or a larger time gap starts a new group. This is a chosen, unvalidated tolerance: a **near-time tile mosaic**, not a verified single acquisition. Our retained metadata cannot establish shared orbit/datatake identity.
- **Overlap priority:** within each group, sort by tile ID, then UUID, ascending—not by time, catalogue result order or QA. For example, in Oetztal `T32TPS` precedes `T32TPT`. UUID is a deterministic tie-breaker with no chronological or quality meaning.
- **Pixel selection:** the first valid snow value wins. A later valid pixel replaces an invalid pixel, but never an already valid one. Valid means FSC 0–100 or SWS 110/125. Before a valid value is found, retain the first non-no-data class; 255 is no-data. Values are not averaged.
- **Matched QA:** always take snow and QA from the same source tile. QA does not determine priority or filter values. This deliberately mixes source pixels within a near-time group; it is not a best-quality composite.
- **Time:** retain earliest and latest source timestamps and show their range in the overview. Summary points use the earliest timestamp as a plotting coordinate, not a per-pixel observation time.

**Why can overlapping tiles differ?**

FSC and SWS tiles are processed independently and are not harmonised across overlaps, so their values or classifications may differ marginally. Monthly water masks, by contrast, are harmonised across tiles.

Whole-AOI summaries retain FSC mean snow fraction and SWS wet / (wet + dry/no/patchy snow). Only SWS receives elevation-band summaries. SWS QA describes wet detections only. Band areas use projected cell area, a local UTM approximation rather than exact equal-area measurement.

</details>

In [ ]:
# RUN ONLY — no changes needed
import re
import xml.etree.ElementTree as ET
from functools import lru_cache

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.axes import Axes
from matplotlib.image import AxesImage
from matplotlib.text import Text
from numpy.typing import ArrayLike
from shapely.geometry import shape


def monthly_ticks(ax: Axes) -> None:
    """Label every month on a time axis."""
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))


start, end = pd.Timestamp(start_date, tz="UTC"), pd.Timestamp(end_date, tz="UTC")
assert start < end, "Start must be earlier than end."
assert all(n is None or (isinstance(n, int) and n > 0) for n in samples_per_month.values())
geometry = aoi.geometry.unary_union
search_settings = {"aoi": geometry.wkt, "start": start_date, "end": end_date}

# Catalogue discovery and OData asset access
def catalogue_rows(query: str) -> Iterator[dict]:
    """Yield products from every public catalogue result page."""
    url, params = catalogue, {"$filter": query, "$top": 1000}
    while url:
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        page = response.json()
        yield from page["value"]
        url, params = page.get("@odata.nextLink"), None


@lru_cache(maxsize=None)
def product_files(product_id: str, endings: tuple[str, ...]) -> dict[str, str]:
    """Find matching asset URLs in a product directory."""
    pending = [(f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/Nodes", 0)]
    found = {}
    while pending:
        url, depth = pending.pop()
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        page = response.json()
        for node in page.get("result", page.get("value", [])):
            name, child = node["Name"], node["Nodes"]["uri"]
            if name.endswith(endings):
                found[name] = child.removesuffix("/Nodes") + "/$value"
            elif depth < 6 and ("." not in name or name.endswith(".SAFE")):
                pending.append((child, depth + 1))
        if page.get("@odata.nextLink"):
            pending.append((page["@odata.nextLink"], depth))
    return found


def cdse_bytes(session: requests.Session, url: str) -> bytes:
    """Download an asset without exposing authentication errors."""
    try:
        response = session.get(url, timeout=180)
        if response.status_code == 401:
            with cdse_session() as fresh:
                session.headers.update(fresh.headers)
            response = session.get(url, timeout=180)
        response.raise_for_status()
        return response.content
    except Exception:
        raise RuntimeError("CDSE asset download failed. Check access and connectivity.") from None

catalogue = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
snow_products = {
    "FSC": dict(dataset="clms_wsi_fractional-snow-cover_europe_utm_20m_daily_v2",
                layer="FSCTOC", resolution=20, color="#287cac"),
    "SWS": dict(dataset="clms_wsi_sar-wet-snow_europe_utm_60m_daily_v2",
                layer="WSM", resolution=60, color="#ad591f"),
}
datasets = {kind: spec["dataset"] for kind, spec in snow_products.items()}
datasets["DEM"] = "COP-DEM_GLO-30-DGED/2024_1"
products = []
for kind, dataset in datasets.items():
    collection, attribute = ("CCM", "dataset") if kind == "DEM" else ("CLMS", "datasetIdentifier")
    query = (f"Collection/Name eq '{collection}' and "
             f"Attributes/OData.CSC.StringAttribute/any(a:a/Name eq '{attribute}' and "
             f"a/OData.CSC.StringAttribute/Value eq '{dataset}') and "
             f"OData.CSC.Intersects(area=geography'SRID=4326;{box(*geometry.bounds).wkt}')")
    if kind != "DEM":
        query += f" and ContentDate/Start ge {start.isoformat()} and ContentDate/Start lt {end.isoformat()}"
    for p in catalogue_rows(query):
        if shape(p["GeoFootprint"]).intersection(geometry).area > 0:
            products.append({"kind": kind, "id": p["Id"], "name": p["Name"],
                             "date": p["ContentDate"]["Start"], "footprint": p["GeoFootprint"]})
inventory = pd.DataFrame(products, columns=["kind", "id", "name", "date", "footprint"]).drop_duplicates("id")

# Discover monthly water masks in the public bucket
bucket = "https://s3.waw3-2.cloudferro.com/HRWSI"
water_records = []
months = pd.period_range(start.tz_localize(None).to_period("M"),
                         (end - pd.Timedelta(nanoseconds=1)).tz_localize(None).to_period("M"), freq="M")
hydro_years = sorted({m.year if m.month >= 9 else m.year - 1 for m in months})
for tile in sorted(water.tile):
    for year in hydro_years:
        params = {"list-type": 2, "prefix": f"WLRA/{tile}/{year}/"}
        while True:
            response = requests.get(bucket, params=params, timeout=120)
            response.raise_for_status()
            listing = ET.fromstring(response.content)
            for item in listing.findall("{*}Contents"):
                key = item.findtext("{*}Key")
                match = re.search(r"_(\d{8})P1M_.*_WaM\.tif$", key)
                if match and pd.Period(match[1][:6], freq="M") in months:
                    water_records.append({"tile": tile, "month": match[1][:6], "key": key})
            token = listing.findtext("{*}NextContinuationToken")
            if not token:
                break
            params["continuation-token"] = token
water_inventory = pd.DataFrame(water_records, columns=["tile", "month", "key"]).drop_duplicates()

### 4.2 · Elevation and hillshade — Copernicus DEM

Where is the study area high or low? The terrain view uses Copernicus DEM.

In [ ]:
# RUN ONLY — no changes needed
from configparser import ConfigParser, Error as ConfigError
from rasterio.io import MemoryFile
from rasterio.transform import array_bounds, from_origin
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from rasterio.warp import reproject
from rasterio.enums import Resampling
import rasterio
from matplotlib.colors import LightSource, Normalize
from pyproj import CRS as PyprojCRS


# CDSE authentication
def cdse_session() -> requests.Session:
    """Authenticate without displaying credentials, tokens or response bodies."""
    config = ConfigParser(interpolation=None)
    try:
        with open("credentials.ini", encoding="utf-8") as credentials_file:
            config.read_file(credentials_file)
        username = config.get("CDSE", "username", fallback="").strip()
        password = config.get("CDSE", "password", fallback="").strip()
        totp = config.get("CDSE", "totp", fallback="").strip()
    except (OSError, ConfigError, UnicodeError):
        raise RuntimeError("Cannot read credentials.ini. Create it beside the notebook using the template in section 1.") from None
    if not username or not password:
        raise RuntimeError("credentials.ini needs non-empty username and password in the [CDSE] section.")
    try:
        payload = {"client_id": "cdse-public", "grant_type": "password",
                   "username": username, "password": password}
        if totp:
            payload["totp"] = totp
        response = requests.post(
            "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
            data=payload, timeout=60,
        )
        response.raise_for_status()
        session = requests.Session()
        session.headers["Authorization"] = "Bearer " + response.json()["access_token"]
        return session
    except Exception:
        raise RuntimeError("CDSE sign-in failed. Check credentials, account access and connectivity.") from None


def dem_url(product_id: str) -> str:
    """Find the single DEM GeoTIFF in a product."""
    files = product_files(product_id, ("_DEM.tif",))
    if len(files) != 1:
        raise ValueError("Expected one DEM GeoTIFF in this product.")
    return next(iter(files.values()))


# Native-grid validation and raster reads
def raster_analysis_crs(src: Any, wlra: bool = False) -> Any:
    """Use the documented WLRA CRS only for verified water-product metadata."""
    if not wlra:
        return src.crs
    if src.crs is None:
        raise NotImplementedError("WLRA raster has no CRS metadata.")
    declared = PyprojCRS.from_wkt(src.crs.to_wkt())
    expected = PyprojCRS.from_epsg(3035)
    known_local = (declared.is_engineering and
                   declared.name == "ETRS89-extended / LAEA Europe" and
                   [axis.direction for axis in declared.axis_info] == ["east", "north"] and
                   all(axis.unit_name == "metre" and axis.unit_conversion_factor == 1
                       for axis in declared.axis_info))
    if not (declared.equals(expected) or known_local):
        raise NotImplementedError("Unexpected WLRA CRS: expected EPSG:3035 or its known LAEA Europe local-CRS representation.")
    return expected


def native_grid(source: str, area: gpd.GeoDataFrame, *, wlra: bool = False) -> tuple[dict, np.ndarray]:
    """Crop the AOI on the unchanged source grid."""
    with rasterio.open(source) as src:
        if src.crs is None or src.transform.b != 0 or src.transform.d != 0:
            raise NotImplementedError("A declared CRS and north-up source grid are required.")
        crs = raster_analysis_crs(src, wlra)
        boundary = area.to_crs(crs).geometry.unary_union
        window = geometry_window(src, [boundary], boundless=True)
        target = dict(crs=crs, transform=src.window_transform(window),
                      width=int(window.width), height=int(window.height))
    if target["width"] * target["height"] > 80_000_000:
        raise ValueError("AOI grid too large for this hands-on workflow.")
    inside = geometry_mask([boundary], (target["height"], target["width"]),
                           target["transform"], invert=True)
    if not inside.any():
        raise ValueError("AOI contains no source-pixel centres.")
    return target, inside


def display_grid(area: gpd.GeoDataFrame, resolution: int = 30) -> tuple[dict, np.ndarray]:
    """Create a local metric grid for terrain display."""
    centre = area.to_crs(4326).geometry.unary_union.centroid
    crs = f"+proj=laea +lat_0={centre.y} +lon_0={centre.x} +datum=WGS84 +units=m"
    boundary = area.to_crs(crs).geometry.unary_union
    left, bottom, right, top = boundary.bounds
    left, top = math.floor(left / resolution) * resolution, math.ceil(top / resolution) * resolution
    width, height = math.ceil((right-left)/resolution), math.ceil((top-bottom)/resolution)
    transform = from_origin(left, top, resolution, resolution)
    inside = geometry_mask([boundary], (height, width), transform, invert=True)
    return dict(crs=crs, transform=transform, width=width, height=height), inside


def read_raster(
    source: str, target: dict, fill: float = 255, *, wlra: bool = False,
) -> np.ndarray:
    """Read aligned native cells without reprojecting or resampling."""
    with rasterio.open(source) as src:
        transform = target["transform"]
        col, row = ~src.transform * (transform.c, transform.f)
        compatible = (raster_analysis_crs(src, wlra) == target["crs"] and
                      np.allclose([src.transform.a, src.transform.b, src.transform.d, src.transform.e],
                                  [transform.a, transform.b, transform.d, transform.e], rtol=0, atol=1e-10) and
                      np.allclose([col, row], np.round([col, row]), rtol=0, atol=1e-5))
        if not compatible:
            raise NotImplementedError("Incompatible source grids: no automatic analysis reprojection. Choose a compatible AOI or explicitly review a destination grid.")
        window = Window(round(col), round(row), target["width"], target["height"])
        dtype = np.dtype(src.dtypes[0])
        if np.isnan(fill) and not np.issubdtype(dtype, np.floating):
            dtype = np.dtype("float64")
        values = np.full((target["height"], target["width"]), fill, dtype=dtype)
        # Clip explicitly: boundless reads with a fill value can override source nodata.
        left, top = max(0, int(window.col_off)), max(0, int(window.row_off))
        right = min(src.width, int(window.col_off + window.width))
        bottom = min(src.height, int(window.row_off + window.height))
        if right > left and bottom > top:
            data = src.read(1, window=Window(left, top, right-left, bottom-top),
                            masked=True, out_dtype=dtype.name)
            row0, col0 = top - int(window.row_off), left - int(window.col_off)
            values[row0:row0 + data.shape[0], col0:col0 + data.shape[1]] = data.filled(fill)
        return values

# Download DEM tiles and prepare terrain
assert search_settings["aoi"] == aoi.geometry.unary_union.wkt, "AOI changed: rerun discovery."
dem_products = inventory[inventory.kind == "DEM"].sort_values("id")
if dem_products.empty:
    raise ValueError("No DEM products found for this AOI.")

grids = {}
with cdse_session() as session:
    try:
        for product in dem_products.itertuples():
            with MemoryFile(cdse_bytes(session, dem_url(product.id))) as source:
                if "DEM" not in grids:
                    grids["DEM"] = native_grid(source.name, aoi)
                    dem_grid, dem_inside = grids["DEM"]
                    native_elevation = np.full(dem_inside.shape, np.nan, dtype="float32")
                tile = read_raster(source.name, dem_grid, np.nan)
            take = np.isfinite(tile) & ~np.isfinite(native_elevation)
            native_elevation[take] = tile[take]
    except NotImplementedError:
        raise
    except Exception:
        raise RuntimeError("DEM loading failed. Check CDSE/CCM access and connectivity.") from None

# A separate metric grid is used only to render terrain and hillshade.
terrain_grid, terrain_inside = display_grid(aoi)
elevation = np.full(terrain_inside.shape, np.nan, dtype="float32")
reproject(native_elevation, elevation, src_transform=dem_grid["transform"],
          src_crs=dem_grid["crs"], src_nodata=np.nan,
          dst_transform=terrain_grid["transform"], dst_crs=terrain_grid["crs"],
          dst_nodata=np.nan, resampling=Resampling.bilinear)
terrain_valid = terrain_inside & np.isfinite(elevation)
if not terrain_valid.any():
    raise ValueError("No valid DEM pixels inside the AOI.")

levels = elevation[terrain_valid]
norm = Normalize(vmin=levels.min(), vmax=levels.max())
cmap = plt.get_cmap("terrain")
# Shade before masking the AOI boundary to avoid artificial edge cliffs.
light = LightSource(azdeg=315, altdeg=45)
hillshade = light.hillshade(np.ma.masked_invalid(elevation), dx=30, dy=30)
relief = light.shade(
    np.ma.masked_invalid(elevation), cmap=cmap, norm=norm,
    dx=30, dy=30, blend_mode="soft",
)
relief[..., 3] = terrain_valid
left, bottom, right, top = array_bounds(
    terrain_grid["height"], terrain_grid["width"], terrain_grid["transform"],
)
fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
ax.imshow(relief, extent=[left, right, bottom, top])
aoi.to_crs(terrain_grid["crs"]).boundary.plot(ax=ax, color="#26383d", linewidth=0.8)
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.65, label="Elevation (m)")
area_label = "Oetztal" if study_area == "oetztal" else study_area.title()
ax.set(title=f"{area_label} · elevation and hillshade", aspect="equal")
ax.set_axis_off()
plt.show()

### 4.3 · Product availability

In [ ]:
# RUN ONLY — no changes needed
# Parse names once; reuse these fields for sampling, grouping and raster access.
snow = inventory[inventory.kind.isin(snow_products)].copy()
parts = snow.name.str.split("_")
snow["tile"] = parts.str[4]
snow["stamp"] = parts.str[5]
snow["platform"] = parts.str[6]
snow["time"] = pd.to_datetime(snow.stamp, utc=True, format="%Y%m%dT%H%M%S")
snow["day"] = snow.time.dt.floor("D")

# Near-time tile mosaics, not verified acquisitions: same product/platform,
# distinct tiles, within 120 s of the first tile (not a rolling window).
# A repeated tile starts a new group; timestamp ties use tile ID then UUID.
acquisitions = []
for (kind, platform), candidates in snow.groupby(["kind", "platform"]):
    current = []
    for row in candidates.sort_values(["time", "tile", "id"]).itertuples():
        same = bool(current) and row.tile not in {p.tile for p in current}
        same = same and (row.time - current[0].time).total_seconds() <= 120
        if current and not same:
            acquisitions.append(pd.DataFrame(current))
            current = []
        current.append(row)
    if current:
        acquisitions.append(pd.DataFrame(current))

dates = snow.sort_values(["kind", "day", "id"]).groupby(["kind", "day"], as_index=False).agg(ids=("id", list))
water_months = water_inventory.groupby("month", as_index=False).agg(keys=("key", list), tiles=("tile", lambda tiles: sorted(set(tiles))))
water_months["complete"] = water_months.tiles.map(lambda tiles: set(tiles) == set(water.tile))

assert search_settings == {"aoi": aoi.geometry.unary_union.wkt, "start": start_date, "end": end_date}, "AOI/dates changed: rerun the search."
assert all(n is None or (isinstance(n, int) and n > 0) for n in samples_per_month.values())
dates["selected"] = False
for (kind, month), group in dates.groupby(["kind", dates.day.dt.strftime("%Y-%m")], sort=True):
    limit = samples_per_month[kind]
    # Independent seeded sampling for each product/month.
    rng = np.random.default_rng(np.random.SeedSequence([seed, ["FSC", "SWS"].index(kind), int(month.replace("-", ""))]))
    bins = np.array_split(group.index.to_numpy(), len(group) if limit is None else min(limit, len(group)))
    dates.loc[[rng.choice(part) for part in bins], "selected"] = True
selected_ids = {pid for ids in dates.loc[dates.selected, "ids"] for pid in ids}
selected_acquisitions = [group for group in acquisitions if group.id.isin(selected_ids).any()]

colors = {kind: spec["color"] for kind, spec in snow_products.items()}
colors["Water"] = "#3492a6"
fig, (ax, counts) = plt.subplots(
    1, 2, figsize=(14, 4.5), sharey=True, layout="constrained",
    gridspec_kw={"width_ratios": [5, 1.2]},
)
for y, kind in enumerate(["FSC", "SWS"]):
    group = dates[dates.kind == kind]
    ax.vlines(group.day, y - .15, y + .15, color=colors[kind], alpha=.2)
    ax.vlines(group.loc[group.selected, "day"], y - .15, y + .15, color=colors[kind])
    counts.barh(y, len(group), height=.45, color=colors[kind], alpha=.2)
    counts.barh(y, group.selected.sum(), height=.45, color=colors[kind])
    counts.annotate(f"{group.selected.sum()} / {len(group)}", (len(group), y),
                    xytext=(5, 0), textcoords="offset points", va="center", fontsize=9)
for row in water_months.itertuples():
    left = max(pd.Timestamp(row.month + "01", tz="UTC"), start)
    right = min(pd.Timestamp(row.month + "01", tz="UTC") + pd.offsets.MonthBegin(), end)
    ax.barh(2, (right - left).total_seconds() / 86400, left=mdates.date2num(left), height=.35,
            color=colors["Water"])
counts.barh(2, len(water_months), height=.45, color=colors["Water"])
counts.annotate(f"{len(water_months)} months", (len(water_months), 2),
                xytext=(5, 0), textcoords="offset points", va="center", fontsize=9)
ax.set(yticks=range(3), yticklabels=["FSC", "SWS", "Water"], xlim=(start, end), ylim=(2.6, -.7))
monthly_ticks(ax)
ax.grid(axis="x", color="#d0d7d8", linewidth=.8)
ax.tick_params(axis="y", length=0)
counts.tick_params(axis="y", left=False, labelleft=False)
counts.set_xlabel("Selected / available dates\nWater: available months", fontsize=9)
counts.set_xlim(0, max(1, dates.groupby("kind").size().max() if len(dates) else 0, len(water_months)) * 1.55)
for panel in (ax, counts):
    for side in ("top", "right", "left"):
        panel.spines[side].set_visible(False)
ax.plot([], [], color="#888888", alpha=.3, label="Available dates")
ax.plot([], [], color="#333333", label="Selected dates")
ax.legend(loc="upper right", ncol=2, fontsize=8)
fig.suptitle(f"{area_label} · product availability and sampling\n{start_date} to {end_date} (end exclusive)",
             fontsize=14, fontweight="bold")
fig.supxlabel("Snow counts: selected / available UTC dates · Water: available monthly products\n"
              "Catalogue availability does not guarantee valid pixels", fontsize=8)
if not water_months.complete.all():
    print("Some monthly water products have missing tiles; comparable water areas will be withheld.")
plt.show()

Available dates are sampled independently by product and month. Observations on the selected dates are retained.

Availability does not guarantee valid pixels inside the study area.

## 5 · Visualise daily snow observations

What differs between the two dates? Compare the full classification for Date A with the simplified views of Dates A and B. Grey in simplified views denotes masked or unavailable pixels. SWS QA applies to wet detections only; QA does not filter snow values.

FSC retains values 0–100; SWS retains wet snow and dry/no/patchy snow. A valid zero is not missing data.

### 5.1 · Choose two daily-map dates — checkpoint 2

The next cell lists dates with both FSC and SWS in the catalogue. These map dates are independent of seasonal sampling. **Catalogue availability does not guarantee valid pixels**; inspect coverage in the maps. A spring/summer contrast is recommended, not required.

In [ ]:
# RUN ONLY — list dates available for both snow products
assert search_settings == {"aoi": aoi.geometry.unary_union.wkt, "start": start_date, "end": end_date}, "AOI/dates changed: rerun discovery."
common_dates = sorted(set(snow.loc[snow.kind == "FSC", "day"]) & set(snow.loc[snow.kind == "SWS", "day"]))
if not common_dates:
    raise ValueError("No dates contain both FSC and SWS. Choose another period or AOI.")
print("Dates with both FSC and SWS (UTC):")
for month in sorted({day.strftime("%Y-%m") for day in common_dates}):
    print(month + ": " + ", ".join(day.strftime("%Y-%m-%d") for day in common_dates if day.strftime("%Y-%m") == month))

<div role="note" style="background-color: #fff3cd; color: #493600; border-left: 6px solid #e0a800; border-radius: 6px; padding: 16px 20px; margin: 12px 0;">
  <div style="font-size: 1.15em; font-weight: 800; color: #493600; margin-bottom: 8px;">✏️ EDIT THE CELL BELOW — CHOOSE TWO DATES</div>
  <p style="margin: 0 0 8px; color: #493600;">Replace both date placeholders with <strong>two different dates from the available-date list above</strong>, using <code>YYYY-MM-DD</code>. Aim for one spring date and one summer date.</p><p style="margin: 0 0 8px; color: #493600;"><strong>Suggested Oetztal dates:</strong> <strong>2019-05-24</strong> and <strong>2019-07-23</strong>, <strong>only if they appear in the list</strong>. Other areas or periods may need different choices.</p><p style="margin: 0 0 8px; color: #493600;"><strong>Then select the cell below and use Run → Run Selected Cell and All Below.</strong> Empty or invalid inputs deliberately stop execution. The next step downloads snow data and may take some time.</p>
</div>

In [ ]:
# EDIT THIS CELL — checkpoint 2: choose two dates from the list above
example_dates = [
    "YYYY-MM-DD",  # <----- ENTER your first date as "YYYY-MM-DD"
    "YYYY-MM-DD",  # <----- ENTER your second date as "YYYY-MM-DD"
]

# CHECKPOINT — leave the assertions below unchanged.
example_days = []
_date_help = (
    " Fill in the two dates in this cell using the available-date list above. "
    "Then select this cell and use Run > Run Selected Cell and All Below to finish."
)
assert isinstance(example_dates, list) and len(example_dates) == 2, (
    "Enter exactly two example dates." + _date_help
)
assert all(isinstance(value, str) and value.strip() for value in example_dates), (
    "Both date inputs must be filled in." + _date_help
)
try:
    _parsed_dates = [date.fromisoformat(value) for value in example_dates]
    assert all(parsed.isoformat() == value for parsed, value in zip(_parsed_dates, example_dates))
except (AssertionError, TypeError, ValueError):
    raise AssertionError("Both dates must be valid dates in YYYY-MM-DD format." + _date_help) from None
assert example_dates[0] != example_dates[1], "Choose two different dates." + _date_help
assert all(start_date <= value < end_date for value in example_dates), (
    "Both dates must be inside the configured window (end excluded)." + _date_help
)
_checked_days = list(pd.to_datetime(example_dates, utc=True, format="%Y-%m-%d"))
assert all(day in common_dates for day in _checked_days), (
    "Both dates must appear in the available-date list above." + _date_help
)
example_days = _checked_days
print("Both dates are valid. Continue with the remaining cells.")

In [ ]:
# RUN ONLY — no changes needed
# Authenticated snow asset access
@lru_cache(maxsize=1)
def snow_session() -> requests.Session:
    """Reuse a private session; cdse_bytes refreshes expired sign-in."""
    return cdse_session()


def snow_url(product: Any, layer: str) -> str:
    """Download an OData node once to a local cache and return its path."""
    if layer not in ("FSCTOC", "FSCTOC-QA", "WSM", "WSM-QA"):
        raise ValueError("Unsupported snow layer.")
    cache = Path("cdse-assets")
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / f"{product.id}_{layer}.tif"
    if not path.is_file():
        files = product_files(product.id, (f"_{layer}.tif",))
        if len(files) != 1:
            raise ValueError(f"Expected one {layer} GeoTIFF for product {product.id}.")
        data = cdse_bytes(snow_session(), next(iter(files.values())))
        temporary = path.with_suffix(".part")
        temporary.write_bytes(data)
        temporary.replace(path)
    return str(path)


# Official CLMS product palettes: https://github.com/copernicus-land/colourmaps-legends/tree/main/colourmaps
# FSCTOC 0–100, in order; remaining classes and QA are explicit RGBA bytes.
_FSC_HEX = (
    "000000 083370 0a3571 0d3773 0f3974 123b76 143d77 173f79 19417a 1c437c "
    "1e467d 21487e 234a80 264c81 284e83 2b5084 2d5286 305487 325689 35588a "
    "375a8b 3a5c8d 3c5e8e 3f6090 416291 446493 466794 496996 4b6b97 4e6d98 "
    "506f9a 53719b 55739d 58759e 5a77a0 5d79a1 5f7ba3 627da4 647fa5 6781a7 "
    "6983a8 6c85aa 6e87ab 718aad 738cae 768eb0 7890b1 7b92b2 7d94b4 8096b5 "
    "8298b7 859ab8 879cba 8a9ebb 8ca0bd 8fa2be 91a4bf 94a6c1 96a8c2 99abc4 "
    "9badc5 9eafc7 a0b1c8 a3b3ca a5b5cb a8b7cc aab9ce adbbcf afbdd1 b2bfd2 "
    "b4c1d4 b7c3d5 b9c5d7 bcc7d8 bec9d9 c1cbdb c3cedc c6d0de c8d2df cbd4e1 "
    "cdd6e2 d0d8e4 d2dae5 d5dce6 d7dee8 dae0e9 dce2eb dfe4ec e1e6ee e4e8ef "
    "e6eaf1 e9ecf2 ebeff3 eef1f5 f0f3f6 f3f5f8 f5f7f9 f8f9fb fafbfc fdfdfe "
    "ffffff"
).split()
OFFICIAL_SNOW_PALETTES = {
    "FSC": {"values": {**{i: tuple(bytes.fromhex(color)) + (255,) for i, color in enumerate(_FSC_HEX)},
                       **{205: (123, 123, 123, 255), 210: (100, 100, 215, 255), 255: (0, 0, 0, 0)}},
            "quality": {0: (93, 164, 0, 255), 1: (189, 189, 91, 255), 2: (255, 194, 87, 255), 3: (255, 70, 37, 255), 205: (123, 123, 123, 255), 210: (100, 100, 215, 255), 255: (0, 0, 0, 0)}},
    "SWS": {"values": {110: (255, 0, 255, 255), 125: (90, 230, 180, 255), 200: (40, 40, 40, 255), 210: (100, 100, 215, 255), 220: (25, 160, 35, 255), 230: (255, 190, 100, 255), 240: (60, 60, 60, 255), 255: (0, 0, 0, 0)},
            "quality": {0: (93, 164, 0, 255), 1: (189, 189, 91, 255), 2: (255, 194, 87, 255), 3: (255, 70, 37, 255), 250: (123, 123, 123, 255), 255: (0, 0, 0, 0)}},
}


# Snow mosaics and summaries
def valid_snow(values: np.ndarray, kind: str) -> np.ndarray:
    """Identify valid snow values using product-specific classes."""
    return np.isin(values, [110, 125]) if kind == "SWS" else (values >= 0) & (values <= 100)


def prepare_snow(group: pd.DataFrame) -> dict:
    """Mosaic one near-time tile group, preserving classes and matched QA."""
    kind = group.kind.iloc[0]
    spec = snow_products[kind]
    target, inside = grids[kind]
    values = np.full(inside.shape, 255, dtype="uint8")
    quality = np.full_like(values, 255)
    palettes = OFFICIAL_SNOW_PALETTES[kind]
    # Tile-first priority, independent of catalogue order and sensing time.
    for product in group.sort_values(["tile", "id"]).itertuples():
        image = read_raster(snow_url(product, spec["layer"]), target)
        qa = read_raster(snow_url(product, spec["layer"] + "-QA"), target)
        previous = valid_snow(values, kind)
        # First valid snow wins; QA follows the same source, not best QA.
        take = valid_snow(image, kind) & ~previous
        take |= (values == 255) & (image != 255)
        values[take], quality[take] = image[take], qa[take]
    return dict(kind=kind, time=group.time.min(), ids=tuple(sorted(group.id)),
                values=values, quality=quality, inside=inside, grid=target, palettes=palettes,
                time_end=group.time.max())


def snow_summary(observation: dict, area: np.ndarray) -> dict:
    """Summarise snow, coverage and QA within an area mask."""
    values, quality, kind = observation["values"], observation["quality"], observation["kind"]
    valid = area & valid_snow(values, kind)
    total, count = int(area.sum()), int(valid.sum())
    value = np.nan
    if count:
        value = 100 * np.mean(values[valid] == 110) if kind == "SWS" else values[valid].mean()
    eligible = valid & (values == 110) if kind == "SWS" else valid
    qa_valid = eligible & np.isin(quality, [0, 1, 2, 3])
    return dict(value=float(value), valid_pixels=count, area_pixels=total,
                coverage_pct=100 * count / total if total else np.nan,
                qa_mean=float(quality[qa_valid].mean()) if qa_valid.any() else np.nan,
                qa_pixels=int(qa_valid.sum()), qa_eligible_pixels=int(eligible.sum()),
                **{f"qa_class_{i}": int(np.count_nonzero(qa_valid & (quality == i))) for i in range(4)})

# Prepare elevation bands and snow grids
from matplotlib.ticker import MaxNLocator

if not isinstance(target_elevation_bands, int) or target_elevation_bands < 1:
    raise ValueError("Choose a positive target number of elevation bands.")
elevations = native_elevation[dem_inside & np.isfinite(native_elevation)]
band_edges = MaxNLocator(nbins=target_elevation_bands, steps=[1, 2, 2.5, 5, 10]).tick_values(
    elevations.min(), elevations.max(),
)
if band_edges[-1] <= elevations.max():
    band_edges = np.append(band_edges, band_edges[-1] + np.diff(band_edges)[-1])

# Both products need native grids for maps and AOI summaries.
for kind, spec in snow_products.items():
    candidates = snow[snow.kind == kind].sort_values(["tile", "time", "id"])
    if candidates.empty:
        raise ValueError(f"No {kind} products available. Choose another period/AOI.")
    grids[kind] = native_grid(snow_url(next(candidates.itertuples()), spec["layer"]), aoi)
    target, inside = grids[kind]
    if not target["crs"].is_projected or target["crs"].linear_units != "metre":
        raise NotImplementedError("Snow analysis currently requires native projected grids in metres.")

# Only SWS needs DEM alignment and elevation-band membership.
target, inside = grids["SWS"]
aligned = np.full(inside.shape, np.nan, dtype="float32")
reproject(native_elevation, aligned, src_transform=dem_grid["transform"],
          src_crs=dem_grid["crs"], src_nodata=np.nan,
          dst_transform=target["transform"], dst_crs=target["crs"],
          dst_nodata=np.nan, resampling=Resampling.nearest)
elevation_bands = np.digitize(aligned, band_edges).astype("int16") - 1
elevation_bands[~inside | ~np.isfinite(aligned)] = -1



def analysis_settings() -> tuple:
    """Identify the spatial setup behind reusable summaries."""
    source_grids = tuple((kind, str(grids[kind][0]["crs"]), tuple(grids[kind][0]["transform"]),
                          grids[kind][0]["width"], grids[kind][0]["height"]) for kind in snow_products)
    return (aoi.geometry.unary_union.wkt, tuple(band_edges), str(snow_products),
            source_grids, tuple(dem_products.id))


def summarise_observation(observation: dict) -> tuple[dict, list[dict]]:
    """Calculate AOI summaries for both products; elevation bands for SWS only."""
    meta = dict(kind=observation["kind"], time=observation["time"],
                time_end=observation["time_end"], source_ids=observation["ids"])
    whole = dict(meta, **snow_summary(observation, observation["inside"]))
    bands = []
    if observation["kind"] == "SWS":
        bands = [dict(meta, lower_m=low, upper_m=high,
                      **snow_summary(observation, elevation_bands == index))
                 for index, (low, high) in enumerate(zip(band_edges[:-1], band_edges[1:]))]
    return whole, bands

from matplotlib.colors import ListedColormap, BoundaryNorm, to_rgba
from matplotlib.patches import Patch
from IPython.display import HTML, display


def raster_extent(target: dict) -> list[float]:
    """Return raster bounds in Matplotlib image order."""
    left, bottom, right, top = array_bounds(target["height"], target["width"], target["transform"])
    return [left, right, bottom, top]


# Prepare example observations and their overview figures
if len(example_days) != 2 or any(day not in common_dates for day in example_days):
    raise ValueError("Choose two available UTC dates with both FSC and SWS in the map-date cell above.")

# In-memory results for this run: reuse example summaries in seasonal processing.
example_observations, processed_snow = {}, {}
summary_settings = analysis_settings()
for day in dict.fromkeys(example_days):
    for kind in ("FSC", "SWS"):
        candidates = sorted([g for g in acquisitions if g.kind.iloc[0] == kind and g.day.iloc[0] == day], key=lambda g: g.time.min())
        index = example_pass[kind]
        if not 0 <= index < len(candidates):
            raise ValueError(f"{kind} {day.date()}: choose a tile-group index below {len(candidates)}.")
        group = candidates[index]
        observation = prepare_snow(group)
        example_observations[day, kind] = observation
        processed_snow[observation["ids"]] = summarise_observation(observation)
        print(f"{kind} · {observation['time']} · {processed_snow[observation['ids']][0]['coverage_pct']:.1f}% valid snow pixels in study area")

class_labels = {
    "FSC": {205: "Cloud / shadow", 210: "Inland water", 255: "No data"},
    "SWS": {110: "Wet snow", 125: "Dry / no / patchy snow",
            200: "Radar geometry mask", 210: "Water", 220: "Forest",
            230: "Urban", 240: "Non-mountain", 255: "No data"},
}


def palette_color(observation: dict, code: int, layer: str = "values") -> tuple | np.ndarray:
    """Return an official product-palette colour or adapted no-data colour."""
    if code == 255:
        return to_rgba("#dedede")
    color = observation["palettes"][layer].get(code, (166, 166, 166, 255))
    return np.array(color) / 255


def finish_map(ax: Axes, target: dict) -> None:
    """Show only the AOI boundary, with no tile outlines."""
    bounds = raster_extent(target)
    aoi.to_crs(target["crs"]).boundary.plot(ax=ax, color="#333b3e", linewidth=0.6)
    ax.set(xlim=bounds[:2], ylim=bounds[2:], aspect="equal")
    ax.set_axis_off()


def draw_snow_map(ax: Axes, observation: dict, simplified: bool, quality: bool = False) -> AxesImage:
    """Draw prepared snow or QA with the chosen masking style."""
    kind, target = observation["kind"], observation["grid"]
    values, inside = observation["values"], observation["inside"]
    valid = inside & valid_snow(values, kind)
    layer = "quality" if quality else "values"
    shown = observation[layer].copy()
    colors = [palette_color(observation, code, layer) for code in range(256)]
    if quality:
        eligible = valid & (values == 110) if kind == "SWS" else valid
        shown[~eligible | ~np.isin(shown, [0, 1, 2, 3])] = 254
    elif simplified:
        shown[~valid] = 254
    colors[254] = to_rgba("#a6a6a6")
    image = ax.imshow(np.ma.masked_where(~inside, shown), extent=raster_extent(target),
                      cmap=ListedColormap(colors), vmin=-0.5, vmax=255.5, interpolation="nearest")
    finish_map(ax, target)
    if quality:
        ax.set_title(f"{kind} quality", fontsize=10)
    else:
        coverage = processed_snow[observation["ids"]][0]["coverage_pct"]
        first, last = observation["time"], observation["time_end"]
        time_label = first.strftime("%Y-%m-%d %H:%M:%S")
        if last != first:
            time_label += " – " + last.strftime("%H:%M:%S" if last.date() == first.date() else "%Y-%m-%d %H:%M:%S")
        ax.set_title(f"{kind} · {time_label} UTC\nValid snow pixels in study area: {coverage:.1f}%", fontsize=10)
        if kind == "FSC":
            fraction_cmap = ListedColormap([palette_color(observation, code) for code in range(101)])
            ax.figure.colorbar(plt.cm.ScalarMappable(norm=Normalize(0, 100), cmap=fraction_cmap),
                              ax=ax, shrink=0.65, label="Snow fraction (%)")
        labels = {code: text for code, text in class_labels[kind].items() if not simplified or code in (110, 125)}
        handles = [Patch(color=palette_color(observation, code), label=text) for code, text in labels.items()]
        if simplified:
            handles.append(Patch(color="#a6a6a6", label="Masked / unavailable"))
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.02),
                  ncol=2, fontsize=7, frameon=False)
    return image


def plot_overview(day: pd.Timestamp, simplified: bool = True) -> None:
    """Compare FSC and SWS with their quality layers in four panels."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 11), layout="constrained")
    for kind, (ax, qax) in zip(("FSC", "SWS"), axes):
        observation = example_observations[day, kind]
        draw_snow_map(ax, observation, simplified)
        draw_snow_map(qax, observation, simplified, quality=True)
        qax.legend(handles=[Patch(color=palette_color(observation, code, "quality"), label=label)
                            for code, label in enumerate(["0 high", "1 medium", "2 low", "3 minimal"])]
                           + [Patch(color="#a6a6a6", label="Unavailable / not applicable")],
                    title="Quality classes", loc="upper center", bbox_to_anchor=(0.5, -0.02),
                    ncol=2, frameon=False, fontsize=7, title_fontsize=8)
    fig.suptitle(f"{area_label} · {day:%Y-%m-%d} · {'simplified masks' if simplified else 'full classification'}", fontsize=14)
    fig.supxlabel("Grey quality pixels are unavailable or not applicable. Quality does not filter snow values.", fontsize=8)
    plt.show()

plot_overview(example_days[0], simplified=False)

plot_overview(example_days[0], simplified=True)

plot_overview(example_days[1], simplified=True)

### 5.2 · A closer look at low coverage · Oetztal

This separate example from **26 April 2021** is outside the suggested 2019 demonstration period:

- **FSC:** clouds and shadows mask much of the area.
- **SWS:** Sentinel-1 radar observations are distributed on Sentinel-2/MGRS tiles, but may not cover the whole tile. Other masks also limit coverage.
- **Interpretation:** grey is masked or unavailable, not snow-free. Wet-snow percentages describe only the valid subset. FSC and SWS were observed at different times.

The example is skipped for other study areas.

In [ ]:
# RUN ONLY — separate Oetztal coverage example
if study_area == "oetztal":
    coverage_day = pd.Timestamp("2021-04-26", tz="UTC")
    coverage_groups = {
        "FSC": pd.DataFrame([
            {"kind": "FSC", "tile": "T32TPT", "id": "e4e2d17c-fa40-451d-b52d-1d8250f35675", "time": pd.Timestamp("2021-04-26T10:27:38Z")},
        ]),
        "SWS": pd.DataFrame([
            {"kind": "SWS", "tile": "T32TPS", "id": "6cb25ec3-c28c-4140-a24f-8e56ef5fca99", "time": pd.Timestamp("2021-04-26T05:18:43Z")},
            {"kind": "SWS", "tile": "T32TPT", "id": "e9dd903f-8adb-46aa-939c-bae127283a4f", "time": pd.Timestamp("2021-04-26T05:18:43Z")},
        ]),
    }
    for kind, group in coverage_groups.items():
        observation = prepare_snow(group)
        example_observations[coverage_day, kind] = observation
        processed_snow[observation["ids"]] = summarise_observation(observation)
    plot_overview(coverage_day, simplified=True)
else:
    print("This fixed coverage example is specific to Oetztal.")

## 6 · Wet snow by elevation

How does wet snow change with elevation and time? To assign an elevation band to each SWS pixel, the DEM is aligned to the native SWS grid using nearest-neighbour resampling. The SWS data remain on their native grid. The next cell prepares the selected observations and can take time.

The heatmap shows wet snow among valid SWS pixels, without a coverage filter. Grey means missing, not zero. The histogram shows how much of the study area falls in each elevation band, using pixels with elevation data; it does **not** show snow-observation coverage.

In [ ]:
# RUN ONLY — no changes needed
assert summary_settings == analysis_settings(), "Spatial/product settings changed: rerun preparation and example observations."
assert search_settings == {"aoi": aoi.geometry.unary_union.wkt, "start": start_date, "end": end_date}, "AOI/dates changed: rerun discovery."

for index, group in enumerate(selected_acquisitions, 1):
    key = tuple(sorted(group.id))
    if key not in processed_snow or processed_snow[key][0].get("status") == "read_error":
        try:
            observation = prepare_snow(group)
            whole, bands = summarise_observation(observation)
            whole["status"] = "ok" if whole["valid_pixels"] else "no_valid_pixels"
            processed_snow[key] = whole, bands
            del observation
        except NotImplementedError:
            raise  # incompatible analysis grids require an explicit decision
        except Exception:
            processed_snow[key] = (dict(kind=group.kind.iloc[0], time=group.time.min(),
                                       time_end=group.time.max(), source_ids=key, value=np.nan, coverage_pct=np.nan,
                                       valid_pixels=np.nan, qa_mean=np.nan, status="read_error"), [])
            print(f"Unavailable: {group.kind.iloc[0]} {group.time.min()} · raster read failed")

selected_results = [processed_snow[tuple(sorted(group.id))] for group in selected_acquisitions]
snow_summary_table = pd.DataFrame([whole for whole, _ in selected_results]).sort_values(["kind", "time"])
band_summary_table = pd.DataFrame([row for _, bands in selected_results for row in bands])
if not snow_summary_table.empty:
    failures = snow_summary_table.get("status", pd.Series(dtype=str)).eq("read_error").sum()
    if failures:
        print(f"{failures} read failures remain missing, not zero; rerun this cell to retry.")

# Draw wet snow by elevation
def plot_elevation() -> None:
    """Show SWS wet snow through time with aligned elevation-band areas."""
    observations = snow_summary_table[snow_summary_table.kind == "SWS"].sort_values("time")
    data = band_summary_table
    fig, (ax, area_ax) = plt.subplots(1, 2, figsize=(14, 5), sharey=True,
                                     gridspec_kw={"width_ratios": [8, 1.4], "wspace": 0.04}, layout="constrained")
    numbers = mdates.date2num(pd.DatetimeIndex(observations.time).to_pydatetime())
    spacing = np.diff(np.unique(numbers))
    half_width = min(5, np.median(spacing) * 0.35) if len(spacing) else 0.4
    for index, row in enumerate(observations.itertuples()):
        frame = data[data.source_ids.map(lambda ids: ids == row.source_ids)].sort_values("lower_m") if not data.empty else data
        if frame.empty:
            continue
        left = max(numbers[index] - half_width, (numbers[index-1] + numbers[index])/2) if index else numbers[index] - half_width
        right = min(numbers[index] + half_width, (numbers[index] + numbers[index+1])/2) if index+1 < len(numbers) else numbers[index] + half_width
        values = frame.value.to_numpy().copy()
        ax.pcolormesh([left, right], band_edges, np.ma.masked_invalid(values[:, None]),
                      cmap="YlOrRd", vmin=0, vmax=100, shading="flat", rasterized=True)
    colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(0, 100), cmap="YlOrRd"),
                            ax=ax, pad=0.01)
    colorbar.set_label("Wet snow / valid classified pixels (%)", rotation=270, labelpad=10)
    ax.set(xlim=(start, end), ylim=(band_edges[0], band_edges[-1]),
           ylabel="Elevation (m)", yticks=band_edges, facecolor="#ececec")
    monthly_ticks(ax)
    ax.grid(axis="y", alpha=0.15)
    transform = grids["SWS"][0]["transform"]
    cell_area_km2 = abs(transform.a * transform.e) / 1e6
    areas = [np.count_nonzero(elevation_bands == index) * cell_area_km2
             for index in range(len(band_edges) - 1)]
    area_ax.barh((band_edges[:-1] + band_edges[1:]) / 2, areas,
                 height=np.diff(band_edges), color="#8197a4", edgecolor="white")
    area_ax.set(xlabel="Area in elevation band (km²)", title="Study area by elevation", xlim=(0, None))
    area_ax.tick_params(axis="y", labelleft=False)
    area_ax.grid(axis="y", alpha=0.15)
    fig.suptitle(f"{area_label} · SWS wet snow by elevation · {start_date} to {end_date} (end exclusive)")
    plt.show()


plot_elevation()

## 7 · Monthly water

Where does mapped water change? Grey means unavailable, not dry land.

The comparison uses common valid support across all requested months. If an input month or tile is missing, comparable water areas are withheld.

In [ ]:
# RUN ONLY — no changes needed
# Water-map display helpers
def water_display(record: dict) -> np.ma.MaskedArray:
    """Prepare display classes without changing water observations."""
    return np.ma.masked_where(~grids["Water"][1], np.where(record["valid"], record["values"], 2))


def draw_water(ax: Axes, record: dict) -> AxesImage:
    """Draw a prepared water mask over shared hillshade."""
    ax.imshow(np.ma.masked_where(~water_inside | ~np.isfinite(water_hillshade), water_hillshade), cmap="gray",
              vmin=0, vmax=1, alpha=0.18, extent=raster_extent(water_grid))
    image = ax.imshow(water_display(record), extent=raster_extent(grids["Water"][0]),
                      cmap=ListedColormap([(1, 1, 1, 0), "#0050cc", "#a6a6a6"]),
                      norm=BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3), interpolation="nearest")
    aoi.to_crs(water_grid["crs"]).boundary.plot(ax=ax, color="#333b3e", linewidth=0.7)
    ax.set_aspect("equal")
    ax.set_axis_off()
    return image


# Read and summarise monthly water
def prepare_water(month: pd.Period) -> dict:
    """Prepare one native monthly water mosaic and its validity."""
    target, inside = grids["Water"]
    files = water_inventory[water_inventory.month == month.strftime("%Y%m")].sort_values(["tile", "key"])
    if files.tile.duplicated().any():
        raise ValueError(f"Multiple water files for one tile in {month}; select a product version first.")
    values = np.full(inside.shape, 255, dtype="uint8")
    failed = []
    for product in files.itertuples():
        try:
            image = read_raster(f"{bucket}/{product.key}", target, wlra=True)
            take = np.isin(image, [0, 1]) & ~np.isin(values, [0, 1])
            take |= (values == 255) & (image != 255)
            values[take] = image[take]
        except NotImplementedError:
            raise
        except Exception:
            failed.append(product.tile)
    valid = inside & np.isin(values, [0, 1])
    complete = set(files.tile) == set(water.tile) and not failed
    return dict(month=month, values=values, valid=valid, complete=complete,
                coverage_pct=100 * valid.sum() / inside.sum(),
                source_keys=tuple(files.key), failed_tiles=tuple(failed))


assert search_settings == {"aoi": aoi.geometry.unary_union.wkt, "start": start_date, "end": end_date}, "AOI/dates changed: rerun discovery."
if water_inventory.empty:
    raise ValueError("No monthly water masks found for this AOI/period.")
first_water = water_inventory.sort_values(["tile", "month"]).iloc[0]
grids["Water"] = native_grid(f"{bucket}/{first_water.key}", aoi, wlra=True)
water_grid, water_inside = grids["Water"]
if water_grid["crs"].to_epsg() != 3035:
    raise ValueError("Expected the native WLRA equal-area EPSG:3035 grid.")

# Reproject the background once for display; water classifications stay native.
water_hillshade = np.full(water_inside.shape, np.nan, dtype="float32")
reproject(hillshade, water_hillshade, src_transform=terrain_grid["transform"],
          src_crs=terrain_grid["crs"], src_nodata=np.nan,
          dst_transform=water_grid["transform"], dst_crs=water_grid["crs"],
          dst_nodata=np.nan, resampling=Resampling.bilinear)
monthly_water = {}
for month in months:
    monthly_water[month] = prepare_water(month)

common_water_support = grids["Water"][1].copy()
for record in monthly_water.values():
    common_water_support &= record["valid"]
water_comparable = all(record["complete"] for record in monthly_water.values()) and common_water_support.any()
water_pixel_km2 = abs(grids["Water"][0]["transform"].a * grids["Water"][0]["transform"].e) / 1e6
water_summary = pd.DataFrame([
    dict(month=month, coverage_pct=record["coverage_pct"], complete=record["complete"],
         common_support_km2=common_water_support.sum() * water_pixel_km2 if water_comparable else np.nan,
         water_area_km2=np.count_nonzero(common_water_support & (record["values"] == 1)) * water_pixel_km2 if water_comparable else np.nan)
    for month, record in monthly_water.items()
])
if not water_comparable:
    print("Comparable water areas withheld: incomplete input months or no common valid support.")

# Animate the prepared monthly masks
from matplotlib.animation import FuncAnimation

fig_water, ax_water = plt.subplots(figsize=(7, 8), layout="constrained")
water_periods = list(monthly_water)
water_overlay = draw_water(ax_water, monthly_water[water_periods[0]])
water_title = ax_water.set_title("")
fig_water.legend(handles=[Patch(color="#0050cc", label="Water detected"),
                          Patch(facecolor="#eeeeee", label="No water detected · hillshade"),
                          Patch(color="#a6a6a6", label="Unavailable")],
                 loc="outside lower center", ncol=2, frameon=False, fontsize=8)


def update_water(index: int) -> tuple[AxesImage, Text]:
    """Update the animation from an already prepared month."""
    month = water_periods[index]
    record = monthly_water[month]
    water_overlay.set_data(water_display(record))
    status = "" if record["complete"] else " · incomplete inputs"
    water_title.set_text(f"{area_label} · {month.strftime('%B %Y')}\nMonthly water detection{status}")
    return water_overlay, water_title


water_animation = FuncAnimation(fig_water, update_water, frames=len(water_periods), interval=1000)
plt.close(fig_water)
display(HTML(water_animation.to_jshtml(default_mode="loop")))

## 8 · Snow and water together

How do the seasonal patterns compare? FSC is the mean snow-cover fraction across valid pixels, including valid zeroes. SWS is the share of valid classified pixels that are wet snow: wet / (wet + dry/no/patchy snow). Monthly water is detected water area over the same valid area in every month.

Different coverage and time scales limit comparison; similar timing does not establish causation.

In [ ]:
# RUN ONLY — no changes needed
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.lines import Line2D

qa_reference = example_observations[example_days[0], "FSC"]
qa_cmap = LinearSegmentedColormap.from_list("snow_qa", [palette_color(qa_reference, i, "quality") for i in range(4)])


def coverage_faces(base: ArrayLike, coverage: ArrayLike) -> np.ndarray:
    """Blend marker colours using absolute valid-coverage percentages."""
    fraction = np.clip(np.asarray(coverage, dtype=float) / 100, 0, 1)
    return 1 - fraction[:, None] * (1 - np.asarray(base))


def draw_snow_series(ax: Axes, kind: str, quality: bool = False) -> None:
    """Draw snow values coloured by product or mosaic QA."""
    data = snow_summary_table[(snow_summary_table.kind == kind) & snow_summary_table.value.notna()]
    color = snow_products[kind]["color"]
    if quality:
        base = qa_cmap(np.clip(data.qa_mean.fillna(0).to_numpy() / 3, 0, 1))[:, :3]
        base[data.qa_mean.isna()] = to_rgb("#888888")
    else:
        base = np.array(to_rgb(color))
    faces = coverage_faces(base, data.coverage_pct)
    ax.scatter(data.time, data.value, s=23, facecolors=faces, edgecolors="#777777", linewidths=0.15)
    ax.set(ylabel="SWS\nWet snow share (%)" if kind == "SWS" else "Mean FSC (%)", ylim=(-3, 103))
    if quality:
        return  # The quality view uses one combined QA/coverage key.
    legend_base = np.array(to_rgb(color))
    handles = [Line2D([], [], marker="o", linestyle="", markeredgecolor="#777777",
                      markeredgewidth=0.5,
                      markerfacecolor=coverage_faces(legend_base, [coverage])[0],
                      label=f"{coverage}%") for coverage in (0, 50, 100)]
    handles = [Line2D([], [], linestyle="", label="Valid snow pixels in study area:")] + handles
    ax.legend(handles=handles, loc="lower right", bbox_to_anchor=(1, 1.02),
              borderaxespad=0, ncol=4, frameon=False, fontsize=7,
              handlelength=1.2, handletextpad=0.4, columnspacing=1)


def format_time_axes(axes: np.ndarray) -> None:
    """Apply common limits, grid lines and monthly ticks."""
    for ax in axes:
        ax.set_xlim(start, end)
        ax.grid(axis="y", alpha=0.2)
    monthly_ticks(axes[-1])

def draw_water_series(ax: Axes) -> None:
    """Plot monthly water areas on common valid support."""
    dates = water_summary.month.dt.to_timestamp()
    widths = dates.dt.days_in_month
    centres = dates + pd.to_timedelta(widths / 2, unit="D")
    ax.bar(centres, water_summary.water_area_km2, width=widths * 0.8, color="#5398b1", alpha=0.85)
    ax.set(ylabel="Monthly mapped\nwater area (km²)", ylim=(0, None))
    support = water_summary.common_support_km2.dropna()
    if len(support):
        ax.text(0.01, 0.91, f"Common valid support: {support.iloc[0]:.2f} km²", transform=ax.transAxes, fontsize=8)
    else:
        ax.text(0.5, 0.5, "Water areas withheld: incomplete inputs or no common support",
                transform=ax.transAxes, ha="center", fontsize=9)



def draw_quality_key(ax: Axes) -> None:
    """Show the exact QA colours and coverage blending used by snow points."""
    coverages = np.array([0, 25, 50, 100])
    for qa in range(4):
        faces = coverage_faces(np.array(qa_cmap(qa / 3)[:3]), coverages)
        ax.scatter(range(4), [qa] * 4, s=55, facecolors=faces,
                   edgecolors="#777777", linewidths=0.35)
    ax.set(xlim=(-.5, 3.5), ylim=(3.5, -.5),
           xticks=range(4), xticklabels=[f"{coverage}%" for coverage in coverages],
           yticks=range(4), yticklabels=["0 high", "1 medium", "2 low", "3 minimal"])
    ax.set_title("Valid snow pixels in study area (%)", fontsize=9, pad=12)
    ax.set_ylabel("Mean QA class", fontsize=9)
    ax.xaxis.tick_top()
    ax.tick_params(which="both", length=0, labelsize=8)
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which="minor", color="#e5e7eb", linewidth=.6)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_snow_water(quality: bool = False) -> None:
    """Compare snow and water, optionally colouring the same snow values by QA."""
    if quality:
        fig = plt.figure(figsize=(13, 10), layout="constrained")
        layout = fig.add_gridspec(len(snow_products) + 1, 2, width_ratios=[8, 1.6])
        axes = np.array([fig.add_subplot(layout[row, 0]) for row in range(len(snow_products) + 1)])
        for ax in axes[1:]:
            ax.sharex(axes[0])
        for ax in axes[:-1]:
            ax.tick_params(labelbottom=False)
        draw_quality_key(fig.add_subplot(layout[:-1, 1]))
    else:
        fig, axes = plt.subplots(len(snow_products) + 1, 1, figsize=(13, 10), sharex=True, layout="constrained")
    for ax, kind in zip(axes, snow_products):
        draw_snow_series(ax, kind, quality=quality)
    draw_water_series(axes[-1])
    format_time_axes(axes)
    view = " · quality view" if quality else ""
    fig.suptitle(f"{area_label} · snow and monthly water{view} · {start_date} to {end_date} (end exclusive)", fontsize=14)
    # Reserve room below the time axis for concise, two-column notes.
    fig.supxlabel(" \n \n ", fontsize=9)
    fig.text(0.08, 0.045, "FSC  ·  Mean of valid pixels, including zeroes", fontsize=9)
    fig.text(0.08, 0.020, "SWS  ·  Wet snow / (wet + dry/no/patchy snow)", fontsize=9)
    fig.text(0.53, 0.045, "Coverage  ·  Lighter points = fewer valid pixels", fontsize=9)
    fig.text(0.53, 0.020, "Water  ·  Same valid area used for every month", fontsize=9)
    plt.show()


plot_snow_water()

**Optional:** the next cell shows the same comparison coloured by mean QA (0 high to 3 minimal). No edits are needed. Coverage fading and the underlying values remain unchanged; SWS QA describes wet detections only. Mean QA summarises ordinal classes; it is not a probability of correctness.

In [ ]:
# RUN ONLY — no changes needed
plot_snow_water(quality=True)

**Rerun guide:** After changing section 3, use **Run → Run All Cells** and recheck the daily dates. After changing section 5 dates, select their input cell and use **Run → Run Selected Cell and All Below**. If an input check fails, correct it and rerun from that checkpoint. A fresh kernel requires a full run. Only the two **EDIT THIS CELL** cells require participant edits.